#### Import Necessary Packages & Configure Spark

In [15]:
from azure.identity import ClientSecretCredential
from msrest.authentication import BasicAuthentication
from delta.tables import DeltaTable
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import lit, current_timestamp, expr, trim, lower
import traceback, json
from functools import reduce
import math

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")         # To handle older version of dates
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY")            # To handle older version of timestamps 
spark.conf.set("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")     # expecting microsecond format
spark.conf.set("spark.stage.maxConsecutiveAttempts", "1")
spark.conf.set("spark.sql.sources.commitProtocolClass",
               "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol")

StatementMeta(, 761bbcdc-c6b1-4c2b-bfb2-b1ebfc9b82c1, 22, Finished, Available, Finished)

#### Initialize Key Vault and JDBC Driver

In [ ]:
KEY_VAULT_NAME = "kv-abc-anlyt-prod-eus-01"                     # Key vault name
KEY_VAULT_URL = f"https://{KEY_VAULT_NAME}.vault.azure.net/"    # Key vault link 

driver = "com.netsuite.jdbc.openaccess.OpenAccessDriver"        # Driver initialization
logger_table = "lh_utils.logs.raw_ingestion_logger"             # Log table 

StatementMeta(, 761bbcdc-c6b1-4c2b-bfb2-b1ebfc9b82c1, 28, Finished, Available, Finished)

#### Initialize JDBC Connection String

In [ ]:
def jdbc_connection_string(source_name):

    # Initialize the JDBC connection string by retrieving the required secrets from Azure Key Vault.
    if source_name.strip().lower() == 'NetSuite':
        secret_source_name = "wh"

    elif source_name.strip().lower() == 'Netsuite1':
        secret_source_name = "N1"

    CLIENT_SECRET_USERNAME   = notebookutils.credentials.getSecret(KEY_VAULT_URL,f"sec-ns-{secret_source_name}-username")
    CLIENT_SECRET_PASSWORD   = notebookutils.credentials.getSecret(KEY_VAULT_URL,f"sec-ns-{secret_source_name}-password")
    CLIENT_SECRET_ACCOUNT_ID = notebookutils.credentials.getSecret(KEY_VAULT_URL,f"sec-ns-{secret_source_name}-account-id")
    CLIENT_SECRET_ROLE_ID    = notebookutils.credentials.getSecret(KEY_VAULT_URL,f"sec-ns-{secret_source_name}-role-id")

    
    jdbc_url = (
            f"jdbc:ns://{CLIENT_SECRET_ACCOUNT_ID}.connect.api.netsuite.com:1708;"
            f"ServerDataSource=NetSuite2.com;"
            f"Encrypted=1;"
            f"NegotiateSSLClose=false;"
            f"CustomProperties=(AccountID={CLIENT_SECRET_ACCOUNT_ID};"
            f"RoleID={CLIENT_SECRET_ROLE_ID};"
            f"Email={CLIENT_SECRET_USERNAME};"
            f"Password={CLIENT_SECRET_PASSWORD})"
    )

    return {"jdbc_url": jdbc_url, "username": CLIENT_SECRET_USERNAME, "password": CLIENT_SECRET_PASSWORD}

StatementMeta(, 761bbcdc-c6b1-4c2b-bfb2-b1ebfc9b82c1, 29, Finished, Available, Finished)

#### JDBC Read

In [20]:
def spark_jdbc_read(sql_query):
    """
    Reads data from a JDBC source using a SQL query and returns a Spark DataFrame.

    Parameters
    ----------
    sql_query : str
        The query or subquery (wrapped for JDBC) passed to the .option("dbtable").

    Returns
    -------
    DataFrame
        Spark DataFrame containing the queried results.

    Raises
    ------
    Exception
        Any exception raised during the JDBC read is displayed and re-raised.
    """
    try:
        print("Reading...")
        df = (
            spark.read.format("jdbc")
            .option("url", connection_string["jdbc_url"])
            .option("driver", driver)
            .option("dbtable", sql_query)
            .option("fetchsize", "5000")      
            .option("user", connection_string["username"])
            .option("password", connection_string["password"])
            .option("queryTimeout", "1200")
            .load()
        )
        # display(df)
        # df.printSchema()
        print("Reading over")
        return df

    except Exception as e:
        display("ERROR while executing JDBC read:")
        display(str(e))
        raise(e)


StatementMeta(, 761bbcdc-c6b1-4c2b-bfb2-b1ebfc9b82c1, 30, Finished, Available, Finished)

#### Batch Ingestion

In [ ]:
def ingest_data_in_batches(connection_string, table, destination_table, min_date, max_date):
    """
    Incrementally ingests data month-by-month and returns the row count
    """

    row_count = 0
    cols = table['required_columns'] if (table['required_columns'] and table['required_columns'].strip().lower() not in ("none", "null") ) else '*'
    display(f"Ingestion | min_date: {min_date} | max_date: {max_date}")
    
    while min_date < max_date:
        print(min_date, max_date)

        dest_table = destination_table.strip().lower()

        # Extract schema.table (last two parts)
        schema_table = ".".join(dest_table.split(".")[-2:])
        print(schema_table)
        if schema_table == "netsuite.transactionline" and min_date.year >= 2025:
            print("if")
            next_batch = (min_date + relativedelta(months=1)).replace(day=1)
        else:
            print("else")
            next_batch = (min_date + relativedelta(months=3)).replace(day=1)

        batch_end = min(next_batch, max_date)
        print(batch_end)
        # format timestamps for JDBC
        start_str = min_date.strftime("%Y-%m-%d %H:%M:%S")
        end_str = batch_end.strftime("%Y-%m-%d %H:%M:%S")

        display(f"start_str: {start_str}")
        display(f"end_str: {end_str}")
        display(f"destination_table: {destination_table}")

        # SQL query for the batch
        sql_query = f"""(
                        SELECT
                            {cols}
                        FROM  
                            {table['source_object']}
                        WHERE {table['source_modified_date_column']} >= {{ts '{start_str}'}} AND 
                              {table['source_modified_date_column']} <  {{ts '{end_str}'}}
                        ) AS t
        """
        
        # print(sql_query)
        # Read the batch
        df_batch = spark_jdbc_read(sql_query)
        df_batch = df_batch.withColumn("__DataLoadedFrom", lit(table['source_name'])) \
                        .withColumn("__DataLoadedAt", expr("current_timestamp() - INTERVAL 5 HOURS"))
        
        display('Read Completed')

        count_batch = df_batch.count()
        row_count += count_batch

        display(f"count batch: {count_batch}")
        
        if count_batch > 0:
            # smart_write(df_batch, destination_table)
            df_batch.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(destination_table)
            display(f"Data written: {destination_table}")

        display(f"Loaded batch: {start_str} → {end_str} | Rows: {count_batch}")

        # Move to next month
        min_date = batch_end

    return row_count


StatementMeta(, 761bbcdc-c6b1-4c2b-bfb2-b1ebfc9b82c1, 31, Finished, Available, Finished)

#### Full Load

In [22]:
def full_load_data(connection_string, table, destination_table, start_time, load_type = "Full"):
    """
    Performs a full data load from the source

    If the source has a 'last modified date' column, data is ingested month-by-month.
    Otherwise, a complete overwrite of the table is done in a single read.

    Logs are written after the ingestion
    """

    status = "Succeeded"
    error_message = ""
    row_count = 0
    max_date = ""

    cols = table['required_columns'] if (table['required_columns'] and table['required_columns'].strip().lower() != "none") else '*'
    sql_full_load_query = f"(SELECT {cols} from {table['source_object']}) as t"

    display(f"sql_full_load_query: {sql_full_load_query}")

    try:
        # If there is no last modified date column then the "if" is executed for full load of data instead of batch wise ingestion
        if table['source_modified_date_column'] is None:
            print("full load if")
            
            full_refresh_df = spark_jdbc_read(sql_full_load_query)

            full_refresh_df = full_refresh_df.withColumn("__DataLoadedFrom", lit(f"{table['source_name']}")) \
                        .withColumn("__DataLoadedAt", expr("current_timestamp() - INTERVAL 5 HOURS"))
            
            count_month = full_refresh_df.count()
            
            #Append to Lakehouse
            full_refresh_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{destination_table}")
            row_count += count_month

            display(f"Loaded full data into {destination_table}")
            return

        else:
            print("full load else")
        # Determine the minimum least modified date
            sql_min_query = f"(select top 1 {table['source_modified_date_column']} from {table['source_object']} where {table['source_modified_date_column']} is not null order by {table['source_modified_date_column']}) as t"
            min_df = spark_jdbc_read(sql_min_query)
            
            min_date = min_df.select(table['source_modified_date_column']).first()[0]
            max_date = datetime.now()

            if DeltaTable.isDeltaTable(spark, destination_table):
                print("Dropping......")
                spark.sql(f"DROP TABLE IF EXISTS {destination_table}")
            
            if (max_date - min_date) > timedelta(days=2): 
                # Defining columns
                print("small")
                row_count = ingest_data_in_batches(connection_string,table, destination_table, min_date, max_date)

            else:
                try:
                    print("try")
                    display(f"Delta: {max_date - min_date}")
                    df = spark_jdbc_read(sql_full_load_query)
                    
                    df = df.withColumn("__DataLoadedFrom", lit(f"{table['source_name']}")) \
                                .withColumn("__DataLoadedAt", expr("current_timestamp() - INTERVAL 5 HOURS"))
                    
                    count_month = df.count()                
                    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(destination_table)

                    display(f"Loaded full data into {destination_table}")
                
                except Exception as e:
                    # To log any failures in the data ingestion
                    status = "Failed"
                    error_message = str(e) + "\n" + traceback.format_exc()
                    print(error_message)
                    raise(e)
    
    
    except Exception as e:
        # To log any failures in the data ingestion
        status = "Failed"
        error_message = str(e) + "\n" + traceback.format_exc()
        raise(e)
    
    finally:
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        # Create log DataFrame
        log_data = [("logs",
                    table['source_name'],
                    table['sink_object_name'],
                    str(row_count),
                    status,
                    error_message,
                    load_type,
                    str(duration),
                    str(start_time),
                    str(max_date),
                    str(end_time))]

        log_data_load(log_data)

StatementMeta(, 761bbcdc-c6b1-4c2b-bfb2-b1ebfc9b82c1, 32, Finished, Available, Finished)

#### Incremental Load

In [23]:
def incremental_load_data(connection_string,table, last_modified_date, destination_table, start_time, load_type = "Incremental"):
    """
    Performs an Incremental load of data ingestion from the source

    Reads the last modified date and performs month-by-month data ingestion starting from that point.
    Logs are recorded after the data ingestion is completed.
    """

    display("Incremental")
    status = "Succeeded"
    error_message = ""
    row_count = 0
    max_date = ""
    cols = table['required_columns'] if (table['required_columns'] and table['required_columns'].strip().lower() != "none") else '*'

    print(cols) 
    try:
        min_date = last_modified_date + timedelta(seconds=1)
        max_date = datetime.now()
       
        if (max_date - min_date) > timedelta(days=90) and table_size != 'small':
            display('Function Called ingest_data_in_batches')   
            row_count = ingest_data_in_batches(connection_string, table, destination_table, min_date, max_date)
       
        else:
            display(f"Delta: {max_date - min_date}")

            start_str = min_date.strftime("%Y-%m-%d %H:%M:%S")
            end_str = max_date.strftime("%Y-%m-%d %H:%M:%S")

            print(start_str, end_str)
            # SQL query for the batch
            sql_query = f"""(
                            SELECT
                                {cols}
                            FROM
                                {table['source_object']}
                            WHERE
                                {table['source_modified_date_column']} >= {{ts '{start_str}'}} AND
                                {table['source_modified_date_column']} <  {{ts '{end_str}'}}
                            )  AS t
            """

            print(sql_query)
            # Read the batch
            df = spark_jdbc_read(sql_query)
            print("Read")
            # display(df)

            df_to_write = df.withColumn("__DataLoadedFrom", lit(table['source_name'])) \
                            .withColumn("__DataLoadedAt", expr("current_timestamp() - INTERVAL 5 HOURS"))
            print("Rows count")
            row_count = df_to_write.count()
            print(row_count, "Writing")
            df_to_write.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(destination_table)
            
        
    except Exception as e:
         # To log any failures in the data ingestion
        status = "Failed"
        error_message = str(e) + "\n" + traceback.format_exc()
        raise(e)
    
    finally:
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        # Create log DataFrame 
        log_data = [("logs",
                     table['source_name'],
                     table['sink_object_name'],
                     str(row_count),
                     status,
                     error_message,
                     load_type,
                     str(duration),
                     str(start_time),
                     str(last_modified_date),
                     str(end_time))]

        log_data_load(log_data)


StatementMeta(, 761bbcdc-c6b1-4c2b-bfb2-b1ebfc9b82c1, 33, Finished, Available, Finished)

#### Log Data Ingestion

In [24]:
def log_data_load(log_data):
    """
    Stores ingestion logs into the table lh_utils.logs.raw_ingestion_logger
    """
    
    log_columns = [
        "logs",
        "source_name",
        "object_name",
        "row_count",
        "status",
        "error_message",
        "load_type",
        "duration",
        "created_date",
        "max_modified_date",
        "loaded_at"
    ]

    log_df = spark.createDataFrame(log_data, log_columns)
    log_df.write.mode("append").saveAsTable(logger_table)

StatementMeta(, 761bbcdc-c6b1-4c2b-bfb2-b1ebfc9b82c1, 34, Finished, Available, Finished)

#### Check Table Ingestion & Retrieve Max Modified Dates

In [25]:
def check_table_existence(lh_name, schema_name, table_name, last_modified_date_column):
    table_exists        = False
    row_count           = 0
    last_modified_date  = '1900-01-01 00:00:00.000000'
    table_path = f"abfss://PROD_RAW@onelake.dfs.fabric.microsoft.com/{lh_name}.Lakehouse/Tables/{schema_name}/{table_name}"

    # Check if table exists and get last modified date
    if DeltaTable.isDeltaTable(spark, table_path):
        table_exists = True

        if last_modified_date_column is None:
            table_exists = False
            last_modified_date = None
        else:
            last_modified_date = spark.sql(f"SELECT MAX({last_modified_date_column}) AS max_date FROM {lh_name}.{schema_name}.{table_name}").collect()[0]['max_date']
            row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {lh_name}.{schema_name}.{table_name}").collect()[0]['row_count']


    output = {
        "row_count": row_count,
        "table_exists": table_exists,
        "last_modified_date": str(last_modified_date)
    }

    return output

StatementMeta(, 761bbcdc-c6b1-4c2b-bfb2-b1ebfc9b82c1, 35, Finished, Available, Finished)

#### Main Function

In [ ]:
# Parsing the meta data to read the source and destination 
table_list = json.loads(table_array)

for table in table_list:
    # Record the ingestion start time for logging
    start_time = datetime.now()
    
    #To checks table existence and retrieves the last modified date if available
    table_existence_data = check_table_existence(table['sink_name'], table['sink_schema_name'], table['sink_object_name'], table['source_modified_date_column'])
    raw_date = table_existence_data["last_modified_date"]

    # If no previous load is detected, set last_modified_date = None (indicating a full load)
    if raw_date is None or raw_date.strip().lower() == 'none':
        last_modified_date = None
    else:
        if '.' in str(raw_date):
            raw_date = str(raw_date).split('.')[0]
        last_modified_date = datetime.strptime(raw_date, "%Y-%m-%d %H:%M:%S")
    

    destination_table = f"{table['sink_name']}.{table['sink_schema_name']}.{table['sink_object_name']}"

    table_size = f"{table['table_size']}"

    connection_string = jdbc_connection_string(table['source_name'])

    if not table_existence_data['table_exists'] or table['source_extract_strategy'] == 'Full':
        # Always perform full load if:
        # - the table doesn't exist, or
        # - the source extract strategy is 'Full'
        display(f"Executing Full Load {destination_table}")
        full_load_data(connection_string, table, destination_table, start_time, table_size)

    else:
        # Otherwise perform incremental load
        display(f"Executing Incremental Load {destination_table}")
        incremental_load_data(connection_string, table, last_modified_date, destination_table, start_time, table_size)
        


#### Stop Spark Session

In [ ]:
mssparkutils.session.stop()

StatementMeta(, da37c17d-cd79-4956-a235-40955a86386f, -1, Cancelled, , Cancelled)